# Limpeza e qualidade da base de solicitações

Este notebook remove linhas repetidas, padroniza categoria, status e prioridade, e calcula as quantidades e médias usando apenas avaliações válidas (de 1 a 5).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

url = "https://raw.githubusercontent.com/Projetar-Solucoes/materiais-trilha/main/semana-04/solicitacoes_semana4.csv"
df_original = pd.read_csv(url)
print('Linhas na base original:', len(df_original))
display(df_original.head())

## 1. Remover linhas repetidas

In [ ]:
linhas_originais = len(df_original)
repetidas = int(df_original.duplicated().sum())
df = df_original.drop_duplicates().copy()
print('Linhas repetidas removidas:', repetidas)
print('Linhas após remover repetidas:', len(df))

## 2. Padronizar categoria, status e prioridade

As categorias são normalizadas para remover espaços e diferenças entre maiúsculas/minúsculas; depois, grafias equivalentes são reunidas. Status e prioridade também são padronizados.

In [ ]:
# Categoria: normalizar texto e unificar grafias equivalentes
categoria_original = df['categoria'].copy()
categoria_normalizada = df['categoria'].astype('string').str.strip().str.lower()
padrao_categoria = {
    'suporte ti': 'Suporte TI',
    'suporte de ti': 'Suporte TI',
    'acesso': 'Acesso',
    'financeiro': 'Financeiro',
    'infraestrutura': 'Infraestrutura',
    'infra': 'Infraestrutura',
    'dúvida': 'Dúvida',
    'duvida': 'Dúvida',
}
df['categoria'] = categoria_normalizada.map(padrao_categoria).fillna(categoria_normalizada.str.capitalize())
categorias_afetadas = int((categoria_original.astype('string').fillna('') != df['categoria'].astype('string').fillna('')).sum())

# Status: remover espaços e padronizar capitalização
status_original = df['status'].copy()
df['status'] = df['status'].astype('string').str.strip().str.lower().str.capitalize()
status_afetados = int((status_original.astype('string').fillna('') != df['status'].astype('string').fillna('')).sum())

# Prioridade: remover espaços e padronizar capitalização
prioridade_original = df['prioridade'].copy()
df['prioridade'] = df['prioridade'].astype('string').str.strip().str.lower().str.capitalize()
prioridade_afetadas = int((prioridade_original.astype('string').fillna('') != df['prioridade'].astype('string').fillna('')).sum())

print('Linhas com categoria alterada/padronizada:', categorias_afetadas)
print('Linhas com status alterado/padronizado:', status_afetados)
print('Linhas com prioridade alterada/padronizada:', prioridade_afetadas)
display(df.head())

## 3. Quantidade por categoria e média de avaliação válida

Avaliações fora do intervalo de 1 a 5 são tratadas como inválidas e não entram no cálculo da média.

In [ ]:
df['avaliacao_num'] = pd.to_numeric(df['avaliacao'], errors='coerce')
avaliacoes_validas = df['avaliacao_num'].between(1, 5)
avaliacoes_invalidas = int((~avaliacoes_validas & df['avaliacao_num'].notna()).sum())

quantidade_categoria = df['categoria'].value_counts(dropna=False).sort_index()
media_avaliacao_categoria = df.loc[avaliacoes_validas].groupby('categoria')['avaliacao_num'].mean().round(2)

print('Quantidade de solicitações por categoria:')
display(quantidade_categoria)
print('Média de avaliação por categoria (somente notas de 1 a 5):')
display(media_avaliacao_categoria)
print('Avaliações inválidas fora da escala 1–5:', avaliacoes_invalidas)

# Conferência obrigatória: a soma por categoria deve ser igual ao total de linhas limpas
soma_categorias = int(quantidade_categoria.sum())
total_base_limpa = len(df)
print('Soma das quantidades por categoria:', soma_categorias)
print('Total de linhas da base limpa:', total_base_limpa)
assert soma_categorias == total_base_limpa, 'ERRO: a soma por categoria não bate com o total da base limpa.'
print('Conferência OK: as quantidades por categoria somam o total da base limpa.')

## 4. Gerar o relatório de qualidade

Esta célula cria `relatorio_qualidade.md` na mesma pasta em que o notebook estiver salvo. Os números são calculados com os dados reais quando você executa o notebook.

In [ ]:
relatorio = f'''# Relatório de qualidade — base de solicitações

| Problema | Linhas afetadas | O que eu fiz |
|---|---:|---|
| Linhas repetidas | {repetidas} | Removi com `drop_duplicates()` |
| Categoria escrita de jeitos diferentes ou fora do padrão | {categorias_afetadas} | Removi espaços, normalizei o texto e unifiquei grafias equivalentes |
| Status fora do padrão | {status_afetados} | Removi espaços e padronizei a capitalização |
| Prioridade fora do padrão | {prioridade_afetadas} | Removi espaços e padronizei a capitalização |
| Avaliações inválidas (fora de 1 a 5) | {avaliacoes_invalidas} | Excluí essas notas do cálculo da média e registrei a decisão; não apaguei a solicitação |

## Conferência

- Linhas na base original: {linhas_originais}
- Linhas repetidas removidas: {repetidas}
- Linhas na base limpa: {total_base_limpa}
- Soma das solicitações por categoria: {soma_categorias}
- A soma por categoria confere com o total da base limpa: **{'Sim' if soma_categorias == total_base_limpa else 'Não'}**

## Decisão sobre avaliações

Usei apenas avaliações numéricas de 1 a 5 para calcular a média por categoria. Avaliações fora desse intervalo foram consideradas inválidas e não entraram no cálculo, mas as solicitações correspondentes foram mantidas na base.
'''

Path('relatorio_qualidade.md').write_text(relatorio, encoding='utf-8')
print('Arquivo relatorio_qualidade.md criado na pasta atual.')
print(relatorio)